# CRISP-DM Phase 4: Modelling — Model 2, Track B
## SVM Baseline — All Features (No Feature Selection)
Pipeline: Cleaned data → Split → One-Hot Encode → StandardScaler → SVC
All encoded features retained. Contrast with notebook 04 (SVM k=7) which
is expected to fail, demonstrating that the full feature space is required
for SVM's distance-based hyperplane calculation to function correctly.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score

df = joblib.load('../data/cleaned_data.pkl')

# Target column contains strings 'yes'/'no' — map to binary integers
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

X = df.drop('personal_loan', axis=1)
y = df['personal_loan']

# Stratified split preserves 85/15 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")

Train: (4800, 11) | Test: (1200, 11)
Train class balance: {0: 0.85, 1: 0.15}


## One-Hot Encoding — Post-Split, Per Column

In [2]:
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)

    dummies_test = pd.get_dummies(X_test[col], prefix=col)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(f"Feature count after encoding: {X_train.shape[1]}")

Feature count after encoding: 15


## Standardisation — Per Column, Fit on Training Data Only
SVM is a distance-based algorithm — standardisation is mandatory.
Without scaling, high-magnitude features dominate spatial calculations.

In [3]:
continuous_cols = ['age', 'yrs_experience', 'family_size',
                   'income', 'mortgage_amt', 'credit_card_spend']

scaler = StandardScaler()

for col in continuous_cols:
    scaler.fit(X_train[col].values.reshape(-1, 1))
    X_train[col] = scaler.transform(X_train[col].values.reshape(-1, 1))
    X_test[col]  = scaler.transform(X_test[col].values.reshape(-1, 1))

print(f"Scaled: {continuous_cols}")

Scaled: ['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend']


## SVM Baseline — All Features, Default Parameters

In [4]:
svm = SVC(random_state=42)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)
f1          = 2 * (precision * recall) / (precision + recall)
specificity = tn / (tn + fp)

print("=== SVM Baseline — All Features ===")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn}  FP: {fp}")
print(f"  FN: {fn}  TP: {tp}")
print(f"\nPrecision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"\nFull classification report:")
print(classification_report(y_test, y_pred))

=== SVM Baseline — All Features ===

Confusion Matrix:
  TN: 986  FP: 34
  FN: 55  TP: 125

Precision:   0.7862
Recall:      0.6944
F1-Score:    0.7375
Specificity: 0.9667

Full classification report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      1020
           1       0.79      0.69      0.74       180

    accuracy                           0.93      1200
   macro avg       0.87      0.83      0.85      1200
weighted avg       0.92      0.93      0.92      1200

